In [ ]:
import os
import numpy as np
import torch
import joblib
from tqdm import notebook as tqdm

In [ ]:
target = 'data/mint' 
os.makedirs(target, exist_ok=True)
cnt = 0
for subset in tqdm.tqdm(['TotalCapture', 'KIT', 'EyesJapanDataset', 'BMLrub', 'BMLmovi'], desc='subset'):
    for item in os.listdir(subset):
        for subj in tqdm.tqdm(os.listdir(f'{subset}/{item}'), desc='subject', leave=False):
            for seq in tqdm.tqdm(os.listdir(f'{subset}/{item}/{subj}'), desc='seq', leave=False):
                if subset in ['TotalCapture', 'KIT', 'BMLrub', 'BMLmovi']:
                    source = f'data/AMASS/{subset}/{subj}/{seq}.npz'
                else:
                    source = f'data/AMASS/Eyes_Japan_Dataset/{subj}/{seq}.npz'
                data = joblib.load(mapping[source])
                if not os.path.exists(f'{subset}/{item}/{subj}/{seq}/muscle_activations.pkl'): continue
                mact = joblib.load(f'{subset}/{item}/{subj}/{seq}/muscle_activations.pkl')
                values = np.array([mact[mkey].values for mkey in muscle_keys])
                timestamps = np.array(list(mact[muscle_keys[0]].keys()))
                gaps = np.where(np.diff(timestamps) > 0.0201)[0]
                lcur = 0
                for i in tqdm.tqdm(gaps, desc='subseq', leave=False):
                    subtime = timestamps[lcur:i + 1]
                    subval  = values[:, lcur:i + 1] 
                    dataidx = np.round(subtime * 30).astype(int)
                    if dataidx[-1] - dataidx[0] < 30: continue
                    val = torch.nn.functional.interpolate(torch.from_numpy(subval[None]), dataidx[-1] - dataidx[0])[0].T.numpy()
                    joblib.dump({
                        'qpos': data['pose'][dataidx[0]:dataidx[-1]],
                        'qvel': data['pvel'][dataidx[0]:dataidx[-1]],
                        'qacc': data['pacc'][dataidx[0]:dataidx[-1]],
                        'jpos': data['joint'][dataidx[0]:dataidx[-1]],
                        'jvel': data['jvel'][dataidx[0]:dataidx[-1]],
                        'jacc': data['jacc'][dataidx[0]:dataidx[-1]],
                        'mpos': data['mkr'][dataidx[0]:dataidx[-1]],
                        'mvel': data['mvel'][dataidx[0]:dataidx[-1]],
                        'macc': data['macc'][dataidx[0]:dataidx[-1]],
                        'beta': data['beta'],
                        'path': data['path'],
                        'mtau': val,
                    }, f'{target}/{cnt}.pkl')
                    cnt += 1